# Chapter 7 – Unsupervised Learning

## 📖 Summary
Chapter 7 explores **unsupervised learning** — finding structure in data *without* labels. Unlike supervised learning, there's no "right answer" to optimize for, making evaluation more nuanced.

### Key Topics Covered:
1. **Principal Component Analysis (PCA)** – dimensionality reduction
2. **K-Means Clustering** – partition-based clustering
3. **Hierarchical Clustering** – agglomerative, dendrograms
4. **Model-Based Clustering (GMM)** – soft cluster assignment
5. **Scaling & Preprocessing** – critical for distance-based methods
6. **Choosing K** – elbow method, silhouette score


## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris, load_wine, make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, adjusted_rand_score
from scipy.cluster.hierarchy import dendrogram, linkage
import warnings; warnings.filterwarnings('ignore')
np.random.seed(42)

# Load wine dataset
wine = load_wine(as_frame=True)
X = wine.data
y = wine.target

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Dataset: {X.shape[0]} samples, {X.shape[1]} features, {len(np.unique(y))} classes")


## 2. Principal Component Analysis (PCA)

In [ ]:
pca = PCA()
pca.fit(X_scaled)

# Explained variance
explained = pca.explained_variance_ratio_
cumulative = np.cumsum(explained)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].bar(range(1, len(explained)+1), explained, color='steelblue', edgecolor='black', alpha=0.8)
axes[0].plot(range(1, len(explained)+1), cumulative, 'ro-', label='Cumulative')
axes[0].axhline(0.9, color='green', linestyle='--', label='90% threshold')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('Scree Plot')
axes[0].legend()

# 2D projection
pca2 = PCA(n_components=2)
X_pca = pca2.fit_transform(X_scaled)

colors = ['steelblue','tomato','mediumseagreen']
for cls in range(3):
    mask = y == cls
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=colors[cls], label=wine.target_names[cls], alpha=0.8, s=60)
axes[1].set_xlabel(f'PC1 ({explained[0]*100:.1f}% var)')
axes[1].set_ylabel(f'PC2 ({explained[1]*100:.1f}% var)')
axes[1].set_title('PCA: 2D Projection of Wine Dataset')
axes[1].legend()

plt.tight_layout()
plt.savefig('ch7_pca.png', dpi=100, bbox_inches='tight')
plt.show()

n_90 = np.argmax(cumulative >= 0.90) + 1
print(f"Components needed for 90% variance: {n_90}")


### 📚 Theory: PCA
PCA finds **orthogonal directions** (principal components) of maximum variance in the data.

Steps:
1. Standardize features (zero mean, unit variance)
2. Compute the **covariance matrix** $\Sigma$
3. Compute **eigenvectors** (principal components) and **eigenvalues** (variance explained)
4. Project data onto top $k$ eigenvectors

$$z = X W_k$$

- PC1 captures the most variance; each subsequent PC captures less
- PCA is sensitive to feature **scale** → always standardize first
- Scree plot shows how many components to retain


## 3. K-Means Clustering

In [ ]:
# Choose K using Elbow & Silhouette
inertias, sil_scores = [], []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(K_range, inertias, 'bo-')
axes[0].set_xlabel('K'); axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method')

axes[1].plot(K_range, sil_scores, 'go-')
axes[1].set_xlabel('K'); axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score')

plt.tight_layout()
plt.savefig('ch7_kmeans_k.png', dpi=100, bbox_inches='tight')
plt.show()

# Final K-Means with k=3
km3 = KMeans(n_clusters=3, random_state=42, n_init=10)
labels_km = km3.fit_predict(X_scaled)
ari = adjusted_rand_score(y, labels_km)
print(f"K-Means (k=3) ARI vs true labels: {ari:.4f}")


### 📚 Theory: K-Means
K-Means partitions data into $K$ clusters by minimizing **within-cluster sum of squares (WCSS)**:

$$\text{WCSS} = \sum_{k=1}^K \sum_{x \in C_k} \|x - \mu_k\|^2$$

**Algorithm:**
1. Initialize $K$ centroids (random or K-Means++)
2. Assign each point to nearest centroid
3. Recompute centroids as cluster means
4. Repeat 2–3 until convergence

**Choosing K:**
- **Elbow method**: plot WCSS vs K, look for the "elbow"
- **Silhouette score**: measures how similar a point is to its cluster vs. others (range: −1 to 1)


## 4. Hierarchical Clustering

In [ ]:
# Use a subset for readability
idx = np.random.choice(len(X_scaled), 50, replace=False)
X_sub = X_scaled[idx]
y_sub = y[idx]

Z = linkage(X_sub, method='ward')

plt.figure(figsize=(14, 5))
dendrogram(Z, labels=y_sub, leaf_font_size=9,
           color_threshold=10)
plt.title('Hierarchical Clustering Dendrogram (Ward linkage)')
plt.xlabel('Sample (true class label)')
plt.ylabel('Distance')
plt.tight_layout()
plt.savefig('ch7_dendrogram.png', dpi=100, bbox_inches='tight')
plt.show()


### 📚 Theory: Hierarchical Clustering
Builds a **tree of clusters** (dendrogram) without specifying K in advance.

**Agglomerative** (bottom-up):
1. Each point starts as its own cluster
2. Merge the two *closest* clusters
3. Repeat until one cluster remains

**Linkage criteria** define distance between clusters:
- **Single linkage**: minimum pairwise distance (creates chains)
- **Complete linkage**: maximum pairwise distance (creates compact clusters)
- **Ward linkage**: minimizes total within-cluster variance (usually best)

Cut the dendrogram at a horizontal line to obtain $K$ clusters.


## 5. Gaussian Mixture Model (GMM)

In [ ]:
gmm = GaussianMixture(n_components=3, covariance_type='full', random_state=42)
gmm.fit(X_scaled)
labels_gmm = gmm.predict(X_scaled)
probs_gmm  = gmm.predict_proba(X_scaled)

ari_gmm = adjusted_rand_score(y, labels_gmm)
print(f"GMM (k=3) ARI vs true labels: {ari_gmm:.4f}")

# Show soft assignments
print("\nSample soft probabilities (first 5 rows):")
print(pd.DataFrame(probs_gmm[:5].round(3), columns=['Cluster 0','Cluster 1','Cluster 2']))


### 📚 Theory: Gaussian Mixture Models
GMM is a **probabilistic** clustering model that assumes data is generated from a mixture of $K$ Gaussian distributions:

$$p(x) = \sum_{k=1}^K \pi_k \mathcal{N}(x | \mu_k, \Sigma_k)$$

Key advantages over K-Means:
- **Soft assignments** — each point has a probability of belonging to each cluster
- **Flexible shapes** — different covariance structures (spherical, diagonal, full)
- Trained via **EM algorithm** (Expectation-Maximization)

Model selection: use **BIC** or **AIC** to choose the number of components.


## 6. Cluster Comparison

In [ ]:
# Compare K-Means vs GMM vs True labels in PCA space
pca2 = PCA(n_components=2)
X_2d = pca2.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
titles = ['True Labels', f'K-Means (ARI={adjusted_rand_score(y,labels_km):.3f})',
          f'GMM (ARI={ari_gmm:.3f})']
label_sets = [y, labels_km, labels_gmm]
cmaps = ['Set1','Set2','Set3']

for ax, title, lbls, cmap in zip(axes, titles, label_sets, cmaps):
    scatter = ax.scatter(X_2d[:,0], X_2d[:,1], c=lbls, cmap=cmap, s=40, alpha=0.8)
    ax.set_title(title)
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')

plt.suptitle('Clustering Comparison (PCA 2D Projection)', y=1.02)
plt.tight_layout()
plt.savefig('ch7_cluster_compare.png', dpi=100, bbox_inches='tight')
plt.show()


## ✅ Chapter 7 Summary

| Method | Type | Key Hyperparameter | Strengths |
|---|---|---|---|
| PCA | Dimensionality Reduction | n_components | Reduces noise, visualization |
| K-Means | Hard Clustering | K | Simple, scalable |
| Hierarchical | Hard Clustering | linkage, cut height | No K needed upfront |
| GMM | Soft Clustering | K, covariance_type | Probabilistic, flexible shapes |

**Key takeaways:**
- Always **standardize** before PCA, K-Means, or hierarchical clustering.
- Use **multiple metrics** (elbow + silhouette + domain knowledge) to choose K.
- **ARI (Adjusted Rand Index)** measures cluster quality when true labels are available.
- Unsupervised results are exploratory — always interpret with domain knowledge.
